# WordMind 字母块与自然拼读数据质量审查

## tl;dr（修复前基线）

- 本报告保留修复前审计结果；可重新运行代码单元获取当前结果。
- 数据库共有 **453 个唯一单词**，全部进入修复前的 `analyzeWordForStudy` 算法。
- 修复前有 **65 个单词（66 个规则命中）**存在高置信的多字母字素/拼读块被拆散问题。
- **129 个单词（134 个关联）**的数据库自然拼读标签与实际字母块输出不一致；其中一部分是正式字素错误，另一部分属于“词族/教学块是否应整体展示”的产品口径问题。
- `Mr` 出现完整性错误：字母块重建为 `mister`，不再等于原词。
- 标签数据另有两处确定错误：`clear` 和 `why`。

## Context & Methods

目标是检查当前生产数据库中的全部单词，确认顺序学习页展示的音节与字母块是否可信。

### Key Assumptions

- 字母块必须按顺序完整重建原单词，不允许增字、漏字。
- 英式自然拼读中的多字母字素应整体展示；优先参考英国教育部材料与 Lancashire Red Rose Letters and Sounds。
- 数据库标签用于表达既有教学意图。标签与算法不一致不一定全部属于语言学错误，因此单独列为“口径不一致”。
- 音节拆分与字母块是不同层次；本审查重点是本次反馈涉及的字母块。

参考：

- UK DfE Year 1 phonics technical report: https://assets.publishing.service.gov.uk/media/5a7f057ee5274a2e8ab49ab9/phonics_2011_technical_report.pdf
- Lancashire Red Rose progression: https://www.lancashire.gov.uk/media/960107/assessment-and-and-progression-red-rose-letters-and-sounds.pdf
- Lancashire planning document: https://www.lancashire.gov.uk/media/937566/1-red-rose-letters-and-sounds-planning-document.pdf

## Data

### 1. Run the reproducible audit

In [1]:
from pathlib import Path
import json
import subprocess
from IPython.display import Markdown, display

REPO_ROOT = Path("/Users/xia/projects/wordmind")
command = [
    str(REPO_ROOT / "node_modules" / ".bin" / "tsx"),
    str(REPO_ROOT / "scripts" / "audit-phonics-blocks.ts"),
]
completed = subprocess.run(
    command, cwd=REPO_ROOT, check=True, capture_output=True, text=True
)
audit = json.loads(completed.stdout)
{"dataset": audit["dataset"], "summary": audit["summary"]}

{'dataset': {'rows': 453, 'distinctWords': 453, 'taggedWords': 334},
 'summary': {'reconstructionFailures': 1,
  'highConfidenceIssueHits': 66,
  'highConfidenceAffectedWords': 65,
  'tagBlockMismatchHits': 134,
  'tagBlockAffectedWords': 129,
  'missingExpectedTags': 11,
  'knownWrongTagAssignments': 2}}

## Results

### 2. High-confidence multi-letter block failures

In [2]:
rows = ["| 拼读块 | 受影响数 | 单词 |", "|---|---:|---|"]
for group in audit["highConfidenceByPattern"]:
    rows.append(
        f"| `{group['name']}` | {group['count']} | {', '.join(group['words'])} |"
    )
display(Markdown("\n".join(rows)))

| 拼读块 | 受影响数 | 单词 |
|---|---:|---|
| `ture` | 4 | adventure, culture, future, picture |
| `air` | 6 | air, chair, fair, hair, pair, stair |
| `ear` | 11 | bear, clear, dear, ear, hear, learn, near, pear, swear, wear, year |
| `ore` | 4 | before, chore, more, store |
| `are` | 3 | care, dare, share |
| `our` | 10 | colour, colourful, favour, favourite, flavour, four, fourteen, humour, neighbour, your |
| `oor` | 3 | door, floor, poor |
| `nk` | 4 | drink, monkey, piggy bank, pink |
| `eigh` | 4 | eight, eighteen, neighbour, weight |
| `ire` | 5 | fire, firefighter, hire, tired, wire |
| `qu` | 5 | queen, question, quick, quiet, quite |
| `str` | 4 | straw, street, string, strong |
| `eir` | 1 | their |
| `ere` | 2 | there, where |

### 3. Integrity and tag-data failures

In [3]:
display(Markdown(
    "**原词无法由字母块重建：**\n\n```json\n"
    + json.dumps(audit["reconstructionFailures"], ensure_ascii=False, indent=2)
    + "\n```\n\n**确定错误的标签：**\n\n```json\n"
    + json.dumps(audit["knownWrongTagAssignments"], ensure_ascii=False, indent=2)
    + "\n```\n\n**缺少预期标签：**\n\n```json\n"
    + json.dumps(audit["missingExpectedTags"], ensure_ascii=False, indent=2)
    + "\n```"
))

**原词无法由字母块重建：**

```json
[
  {
    "word": "mr",
    "blocks": "m|i|s|t|er",
    "reconstructed": "mister"
  }
]
```

**确定错误的标签：**

```json
[
  {
    "word": "clear",
    "currentTag": "ea/iː/",
    "expectedTag": "ear /ɪə/",
    "reason": "clear uses the ear spelling for /ɪə/ in the configured British pronunciation"
  },
  {
    "word": "why",
    "currentTag": "wh /h/",
    "expectedTag": "wh /w/",
    "reason": "why begins with /w/, not /h/"
  }
]
```

**缺少预期标签：**

```json
[
  {
    "word": "clear",
    "pattern": "ear",
    "tags": [
      "ea/iː/"
    ]
  },
  {
    "word": "colourful",
    "pattern": "our",
    "tags": [
      "or 在非重读音节/ər/"
    ]
  },
  {
    "word": "drink",
    "pattern": "nk",
    "tags": [
      "dr"
    ]
  },
  {
    "word": "eight",
    "pattern": "eigh",
    "tags": [
      "gh不发音"
    ]
  },
  {
    "word": "learn",
    "pattern": "ear",
    "tags": []
  },
  {
    "word": "monkey",
    "pattern": "nk",
    "tags": []
  },
  {
    "word": "neighbour",
    "pattern": "eigh",
    "tags": [
      "our/ə/",
      "gh不发音"
    ]
  },
  {
    "word": "pink",
    "pattern": "nk",
    "tags": []
  },
  {
    "word": "straw",
    "pattern": "str",
    "tags": [
      "aw/ɔː/"
    ]
  },
  {
    "word": "weight",
    "pattern": "eigh",
    "tags": [
      "gh不发音"
    ]
  },
  {
    "word": "where",
    "pattern": "ere",
    "tags": [
      "wh /w/"
    ]
  }
]
```

### 4. All tag-to-block inconsistencies

In [4]:
rows = ["| 数据库标签 | 不一致数 | 单词 |", "|---|---:|---|"]
for group in audit["tagBlockByTag"]:
    rows.append(
        f"| {group['name']} | {group['count']} | {', '.join(group['words'])} |"
    )
display(Markdown("\n".join(rows)))

| 数据库标签 | 不一致数 | 单词 |
|---|---:|---|
| ture/tʃə/ | 4 | adventure, culture, future, picture |
| air | 6 | air, chair, fair, hair, pair, stair |
| all | 8 | all, ball, basketball, call, fall, football, small, tall |
| al | 4 | also, always, talk, walk |
| ong/ɒŋ/ | 5 | among, long, song, strong, wrong |
| ple | 3 | apple, people, purple |
| and | 4 | band, hand, land, stand |
| ask 词族 a /ɑː/ | 1 | basketball |
| ear /eə/ | 4 | bear, pear, swear, wear |
| ore | 4 | before, chore, more, store |
| oot/ood/ook | 7 | book, cook, football, good, hook, look, look after |
| ing/ɪŋ/ | 4 | bring, king, sing, wing |
| uy /aɪ/ | 1 | buy |
| are /eə/ | 3 | care, dare, share |
| ese | 4 | cheese, chinese, japanese, these |
| i+ld/nd | 5 | child, find, kind, mild, wild |
| ass/ɑː/ | 5 | class, classmate, glass, grass, in class |
| old | 4 | cold, gold, hold, old |
| our/ə/ | 6 | colour, favour, favourite, flavour, humour, neighbour |
| ear /ɪə/ | 5 | dear, ear, hear, near, year |
| wor /wɜː/ | 3 | delivery worker, factory worker, office worker |
| oor | 3 | door, floor, poor |
| gh不发音 | 4 | eight, neighbour, right, weight |
| eigh /eɪ/ | 1 | eighteen |
| en /ən/（非重读词尾） | 2 | eleven, seventeen |
| ile | 4 | file, pile, smile, tile |
| ire | 5 | fire, firefighter, hire, tired, wire |
| our /ɔː/ | 3 | four, fourteen, your |
| -ful /fl/ | 1 | helpful |
| ea /ɪə/ | 1 | idea |
| Ms /mɪz/ | 1 | ms |
| eo /iː/ | 1 | people |
| nk /ŋk/ | 1 | piggy bank |
| ice在多音节词 | 1 | police |
| ice 在多音节词中 /iːs/ | 1 | police officer |
| qu | 5 | queen, question, quick, quiet, quite |
| or/ɔː/ | 1 | story |
| str | 3 | street, string, strong |
| ere/eir /eə/ | 2 | their, there |
| war /wɔː/ | 1 | warm |
| wor | 3 | word, work, world |

## 修复结果

1. 已建立“最长匹配优先”的字素表，覆盖 `air / are / ear / ere / eir / eigh / ire / ore / our / oor / qu / nk / str / ture`；例如 `eigh` 优先于 `igh`，`are` 优先于 `ar`。
2. 已把 `Mr` 的发音音节与拼写字母块分开，字母块重新拼接后严格等于原词。
3. 已修正 `clear`、`why` 两处错误标签，并补齐 11 个审计发现的缺失关联。
4. 修复后复审：原词重建失败 **0**、高置信拆分错误 **0**、缺失预期标签 **0**、明确错误标签 **0**。
5. 剩余标签/字母块差异主要是词族教学口径（如 `all、and、old、ong、ile、ple`），未在没有明确产品规则时自动合并。